# E2 — Cross-modal demo: image in / text out, text in / images out

Uses the E1 adapter (DINOv2 -> bge-m3) to run both directions of search
against one gallery.

**What this is:** retrieval. The adapter maps an image's *address* into the
text encoder's space, so nearest-neighbour search crosses the modality
boundary. **What this is not:** generation. Nothing here writes a caption
or draws a picture — it selects from a fixed gallery.

**Why one matrix serves both directions.** Everything ends up in bge-m3's
space: gallery images go through DINOv2 then W, gallery captions and typed
queries go through bge-m3 natively. Once every vector lives in one space
and is L2-normalized, search is a dot product, and it does not care which
modality produced which vector.

**Expected quality, honestly.** E1 measured R@1 = 0.358 against a 1,000
gallery, i.e. the *exact* right item first about a third of the time.
Demo perception is more forgiving — "is the top image plausibly a cat" is
an easier bar than "is it the one specific cat photo" — so results will
look better than that number suggests. Both statements are true; quote the
measured one.

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
os.environ["DATA_DIR"] = "/content/drive/MyDrive/convergence_experiment"

In [ ]:
!pip -q install torch torchvision transformers sentence-transformers pillow

In [ ]:
import numpy as np, torch, json, io, zipfile, urllib.request
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from PIL import Image

DATA_DIR = Path(os.environ["DATA_DIR"])
DEV = "cuda" if torch.cuda.is_available() else "cpu"
IMG_MODEL, TXT_MODEL = "facebook/dinov2-base", "BAAI/bge-m3"
GALLERY_N = 500          # images in the demo gallery
ALPHA = 1e-2

def l2n(V):
    return V / (np.linalg.norm(V, axis=-1, keepdims=True) + 1e-12)

def ridge(X, Y, a=ALPHA):
    return np.linalg.solve(X.T @ X + a * np.eye(X.shape[1]), X.T @ Y)

# W: load if E1 saved it, else refit from the saved pairs (seconds)
wp = DATA_DIR / "crossmodal_adapter.npz"
if wp.exists():
    W = np.load(str(wp))["W"]
    print("loaded W", W.shape)
else:
    d = np.load(str(DATA_DIR / "crossmodal_pairs.npz"))
    W = ridge(d["img"].astype(np.float64), d["txt"].astype(np.float64))
    np.savez_compressed(str(wp), W=W)
    print("refitted W from crossmodal_pairs.npz:", W.shape)

## 1. Build the demo gallery (images + their captions + URLs)

In [ ]:
ANN = DATA_DIR / "annotations_trainval2017.zip"
if not ANN.exists():
    urllib.request.urlretrieve(
        "http://images.cocodataset.org/annotations/"
        "annotations_trainval2017.zip", str(ANN))
with zipfile.ZipFile(ANN) as z:
    with z.open("annotations/captions_val2017.json") as f:
        ann = json.load(f)          # val2017: unseen by the E1 fit

caps = {}
for a in ann["annotations"]:
    caps.setdefault(a["image_id"], []).append(a["caption"].strip())
urls = {im["id"]: im["coco_url"] for im in ann["images"]}
gids = sorted(set(caps) & set(urls))[:GALLERY_N]
print(f"gallery: {len(gids)} images from val2017")

def grab(i):
    try:
        with urllib.request.urlopen(urls[i], timeout=8) as r:
            return i, Image.open(io.BytesIO(r.read())).convert("RGB")
    except Exception:
        return i, None

with ThreadPoolExecutor(max_workers=32) as pool:
    got = [(i, im) for i, im in pool.map(grab, gids) if im is not None]
gal_ids = [i for i, _ in got]
gal_ims = [im for _, im in got]
print(f"fetched {len(gal_ims)}")

In [ ]:
from transformers import AutoImageProcessor, AutoModel
from sentence_transformers import SentenceTransformer

proc = AutoImageProcessor.from_pretrained(IMG_MODEL)
vis = AutoModel.from_pretrained(IMG_MODEL).to(DEV).eval()
if DEV == "cuda":
    vis = vis.half()

def encode_images(images, bs=32):
    out = []
    for b in range(0, len(images), bs):
        with torch.no_grad():
            x = proc(images=images[b:b + bs], return_tensors="pt").to(DEV)
            if DEV == "cuda":
                x["pixel_values"] = x["pixel_values"].half()
            out.append(vis(**x).last_hidden_state[:, 0].float().cpu().numpy())
    return np.concatenate(out).astype(np.float64)

txt_model = SentenceTransformer(TXT_MODEL, device=DEV)
def encode_text(strings):
    return txt_model.encode(strings, batch_size=64, convert_to_numpy=True
                            ).astype(np.float64)

# gallery images -> DINOv2 -> W -> text space
GAL_IMG = l2n(encode_images(gal_ims) @ W)
# gallery captions -> bge-m3 (first caption each, for display)
gal_caps = [caps[i][0] for i in gal_ids]
GAL_TXT = l2n(encode_text(gal_caps))
print("gallery ready:", GAL_IMG.shape, GAL_TXT.shape,
      "- both in bge-m3 space")

## 2. Image in -> text out

Encode the image with DINOv2, push it through W, and rank the gallery
captions by cosine. The image never touches the text model.

In [ ]:
from IPython.display import display

def image_to_text(image, k=5):
    v = l2n(encode_images([image]) @ W)[0]
    s = GAL_TXT @ v
    top = np.argsort(-s)[:k]
    display(image.resize((320, int(320 * image.height / image.width))))
    print("closest captions in the gallery:")
    for r, j in enumerate(top, 1):
        print(f"  {r}. [{s[j]:+.3f}]  {gal_caps[j]}")
    return [(gal_caps[j], float(s[j])) for j in top]

_ = image_to_text(gal_ims[7])      # a gallery image, as a smoke test

In [ ]:
# your own image
from google.colab import files
up = files.upload()
for name, blob in up.items():
    print(f"\n=== {name} ===")
    _ = image_to_text(Image.open(io.BytesIO(blob)).convert("RGB"))

## 3. Text in -> images out

Encode the query with bge-m3 and rank the gallery *images* — which are
sitting in the same space because W put them there. The text never touches
the image model.

Note on query style: the gallery captions are full sentences ("a cat
sitting on a windowsill"), so a bare word like "cat" is out of
distribution for the encoder. Both forms are shown below; the sentence
form usually ranks better, and that gap is itself worth noticing.

In [ ]:
def text_to_image(query, k=5):
    q = l2n(encode_text([query]))[0]
    s = GAL_IMG @ q
    top = np.argsort(-s)[:k]
    print(f'query: "{query}"')
    for r, j in enumerate(top, 1):
        print(f"  {r}. cosine {s[j]:+.3f}")
        display(gal_ims[j].resize((200, int(200 * gal_ims[j].height /
                                            gal_ims[j].width))))
    return [(gal_ids[j], float(s[j])) for j in top]

_ = text_to_image("cat", k=3)

In [ ]:
_ = text_to_image("a cat sitting on a windowsill", k=3)

In [ ]:
for q in ["a person riding a horse on the beach",
          "several people playing baseball",
          "a plate of food on a wooden table"]:
    _ = text_to_image(q, k=2)
    print("-" * 50)

## 4. What the demo does and does not show

- It **does** show that one fitted matrix makes two independently trained,
  single-modality encoders interoperable well enough to search across the
  boundary — DINOv2 has never seen a word, bge-m3 has never seen a pixel.
- It **does not** show generation, understanding, or ceiling-level
  retrieval. The measured figure is R@1 = 0.358 against a 1,000-item
  gallery (40.9% of a jointly trained model's score on the same task);
  demo perception is more forgiving than exact-match recall.
- A **negative control** worth running when showing this: skip W (compare
  raw DINOv2 vectors to bge-m3 vectors, zero-padded) and the ranking
  collapses to chance. Everything you see is created by the matrix.

In [ ]:
# negative control: the same query without the adapter
RAW = np.zeros((len(gal_ims), GAL_TXT.shape[1]))
_r = encode_images(gal_ims)
RAW[:, :_r.shape[1]] = _r
RAW = l2n(RAW)
q = l2n(encode_text(["a cat sitting on a windowsill"]))[0]
print("WITHOUT the adapter, top-3 cosines:",
      np.sort(RAW @ q)[::-1][:3].round(3))
print("WITH    the adapter, top-3 cosines:",
      np.sort(GAL_IMG @ q)[::-1][:3].round(3))
print("\nthe raw ranking is arbitrary - the capability is the matrix")